# 📦 Project Task: SiCepat Ekspres — First-Mile Logistics Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

### Konteks Bisnis
SiCepat Ekspres merebut hati jutaan seller UMKM melalui layanan First-Mile Pickup gratis dan paket murah HALU. Namun pertumbuhan masif ini memunculkan dua masalah kritis: **Phantom Pickup** oleh kurir yang manipulasi data untuk hindari denda KPI, dan **revenue leakage** miliaran rupiah akibat seller yang sengaja memperkecil berat paket di aplikasi.

Kamu berperan sebagai Data Analyst di tim **Business Intelligence SiCepat** yang diminta untuk membersihkan data operasional, mengidentifikasi pola Phantom Pickup dan manipulasi berat, serta memberikan rekomendasi berbasis data untuk SLA enforcement dan revenue recovery.

**Dataset (3 tabel):**
- `sicepat_sellers.csv` — 15.000 baris (Dimensi Seller)
- `sicepat_services.csv` — 5 baris (Dimensi Layanan)
- `sicepat_pickups.csv` — 300.000 baris (Fakta Pickup & Berat)

---

### 🚨 Business Context Error (Sudah Diinvestigasi)
> **10.199 transaksi** dengan `pickup_status = 'Success'` memiliki `pickup_time` yang terjadi **SEBELUM** `request_time`.
> Ini adalah **Phantom Pickup**: kurir menekan tombol 'Pickup Selesai' sebelum seller bahkan memanggil.
> Logika mustahil secara operasional — diidentifikasi, dikuantifikasi, dan direkomendasikan mekanisme deteksi otomatis di Section 2.6 dan 4.4.

---
## 0. Import & Load Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

**📌 Catatan teknis:** ketiga file CSV ternyata menggunakan separator titik koma (`;`), bukan koma. Ini diketahui setelah mencoba load dengan separator default dan hasilnya semua kolom "menggabung" jadi satu kolom string. Maka dari itu kita load dengan `sep=';'`.

In [2]:
# Load semua dataset
# Catatan: file CSV ini menggunakan separator ';' (titik koma), bukan ','
df_sellers_raw  = pd.read_csv('sicepat_sellers.csv', sep=';')
df_services_raw = pd.read_csv('sicepat_services.csv', sep=';')
df_pickups_raw  = pd.read_csv('sicepat_pickups.csv', sep=';')

# Buat copy untuk dikerjakan, agar data mentah asli tetap aman sebagai pembanding
df_sellers  = df_sellers_raw.copy()
df_services = df_services_raw.copy()
df_pickups  = df_pickups_raw.copy()

print('=== SELLERS ===')
display(df_sellers.head())

print('\n=== SERVICES ===')
display(df_services)

print('\n=== PICKUPS ===')
display(df_pickups.head())

=== SELLERS ===


,seller_id,seller_name,city,join_date
0,SEL-00001,Toko Seller Sukses 0,Cimahi,27/11/2021
1,SEL-00002,Toko Seller Sukses 1,Makassar,19/02/2022
2,SEL-00003,Toko Seller Sukses 2,Cimahi,20/06/2021
3,SEL-00004,Toko Seller Sukses 3,Bandung,18/09/2021
4,SEL-00005,Toko Seller Sukses 4,Medan,09/06/2021



=== SERVICES ===


,service_code,service_name,Unnamed: 2
0,SVC-01,HALU,NaN
1,SVC-02,BEST,NaN
2,SVC-03,SIUNTUNG,NaN
3,SVC-04,GOKIL,NaN
4,SVC-05,HALU-COD,NaN



=== PICKUPS ===


,resi_no,seller_id,service_code,item_category,request_time,pickup_time,stated_weight_kg,actual_volume_weight_kg,pickup_status
0,000SC000000001,SEL-04246,SVC-05,ksmtk,20/08/2023 15:00,2023-08-20 20:11:58.720564,2.51,2.98,Success
1,000SC000000002,SEL-04881,SVC-02,Pakaian,14/08/2023 06:00,2023-08-15 01:43:18.472104,-1.50,11.39,Success
2,000SC000000003,SEL-10176,SVC-02,Skincare,04/09/2023 09:00,2023-09-05 01:54:57.252587,4.00,4.04,Success
3,000SC000000004,SEL-05400,SVC-02,Kosmetik,10/11/2023 13:00,2023-11-11 04:30:11.251160,4.74,5.16,Rescheduled
4,000SC000000005,SEL-08010,SVC-02,NaN,03/10/2023 08:00,2023-10-04 07:26:21.201376,4.40,11.69,Success


In [3]:
# df_services punya 1 kolom kosong tambahan ('Unnamed: 2') akibat trailing ';' di file CSV
# Kolom ini tidak berguna, kita buang
print(df_services.columns.tolist())
df_services = df_services.drop(columns=[col for col in df_services.columns if 'Unnamed' in col])
df_services

['service_code', 'service_name', 'Unnamed: 2']


,service_code,service_name
0,SVC-01,HALU
1,SVC-02,BEST
2,SVC-03,SIUNTUNG
3,SVC-04,GOKIL
4,SVC-05,HALU-COD


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [4]:
# Shape dan info umum — lakukan untuk ketiga tabel
print('=== SHAPE ===')
print(f'sellers  : {df_sellers.shape}')
print(f'services : {df_services.shape}')
print(f'pickups  : {df_pickups.shape}')

print('\n=== INFO SELLERS ===')
df_sellers.info()

print('\n=== INFO SERVICES ===')
df_services.info()

print('\n=== INFO PICKUPS ===')
df_pickups.info()

=== SHAPE ===
sellers  : (15000, 4)
services : (5, 2)
pickups  : (300000, 9)

=== INFO SELLERS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   seller_id    15000 non-null  object
 1   seller_name  15000 non-null  object
 2   city         15000 non-null  object
 3   join_date    15000 non-null  object
dtypes: object(4)
memory usage: 468.9+ KB

=== INFO SERVICES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   service_code  5 non-null      object
 1   service_name  5 non-null      object
dtypes: object(2)
memory usage: 212.0+ bytes

=== INFO PICKUPS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 9 columns):
 #   Column                   Non-Nul

In [5]:
# Tipe data seluruh kolom
print('=== TIPE DATA SELLERS ===')
print(df_sellers.dtypes)

print('\n=== TIPE DATA SERVICES ===')
print(df_services.dtypes)

print('\n=== TIPE DATA PICKUPS ===')
print(df_pickups.dtypes)

=== TIPE DATA SELLERS ===
seller_id      object
seller_name    object
city           object
join_date      object
dtype: object

=== TIPE DATA SERVICES ===
service_code    object
service_name    object
dtype: object

=== TIPE DATA PICKUPS ===
resi_no                     object
seller_id                   object
service_code                object
item_category               object
request_time                object
pickup_time                 object
stated_weight_kg           float64
actual_volume_weight_kg    float64
pickup_status               object
dtype: object


In [6]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values

def ringkas_missing(df, nama):
    ringkasan = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing (%)': (df.isnull().sum() / len(df) * 100).round(2)
    })
    ringkasan = ringkasan[ringkasan['Missing Count'] > 0].sort_values('Missing (%)', ascending=False)
    print(f'=== MISSING VALUES — {nama} ===')
    if len(ringkasan) == 0:
        print('Tidak ada missing value.')
    else:
        display(ringkasan)
    print()

ringkas_missing(df_sellers, 'SELLERS')
ringkas_missing(df_services, 'SERVICES')
ringkas_missing(df_pickups, 'PICKUPS')

=== MISSING VALUES — SELLERS ===
Tidak ada missing value.

=== MISSING VALUES — SERVICES ===
Tidak ada missing value.

=== MISSING VALUES — PICKUPS ===


,Missing Count,Missing (%)
item_category,30009,10.00
pickup_time,22448,7.48


In [7]:
# Distribusi kolom-kolom kritis
# stated_weight_kg, actual_volume_weight_kg, pickup_status, item_category

print('=== stated_weight_kg ===')
print(df_pickups['stated_weight_kg'].describe())

print('\n=== actual_volume_weight_kg ===')
print(df_pickups['actual_volume_weight_kg'].describe())

print('\n=== pickup_status ===')
print(df_pickups['pickup_status'].value_counts())

print('\n=== item_category (termasuk NaN) ===')
print(df_pickups['item_category'].value_counts(dropna=False))

=== stated_weight_kg ===
count   300000.00
mean         6.15
std         58.34
min         -1.50
25%          1.61
50%          2.75
75%          3.88
max        999.90
Name: stated_weight_kg, dtype: float64

=== actual_volume_weight_kg ===
count   300000.00
mean         4.15
std          2.84
min          0.50
25%          2.17
50%          3.55
75%          4.88
max         14.99
Name: actual_volume_weight_kg, dtype: float64

=== pickup_status ===
pickup_status
Success        254992
Failed          30090
Rescheduled     14918
Name: count, dtype: int64

=== item_category (termasuk NaN) ===
item_category
Baju          45250
Kosmetik      44985
baju          30040
NaN           30009
Fashion       29942
Pakaian       29874
Skincare      29802
Elektronik    15087
ksmtk         15003
hp            14975
Sepatu        12025
spt            3008
Name: count, dtype: int64


In [8]:
# Cek konsistensi relasi antar tabel
# Apakah semua service_code di pickups ada di services?
# Apakah semua seller_id di pickups ada di sellers?
print('=== KONSISTENSI RELASI ANTAR TABEL ===')

relasi_service = df_pickups['service_code'].isin(df_services['service_code'])
print(f'Semua service_code di pickups ada di services? {relasi_service.all()}')

relasi_seller = df_pickups['seller_id'].isin(df_sellers['seller_id'])
print(f'Semua seller_id di pickups ada di sellers? {relasi_seller.all()}')

=== KONSISTENSI RELASI ANTAR TABEL ===
Semua service_code di pickups ada di services? True
Semua seller_id di pickups ada di sellers? True


**✍️ Ringkasan Temuan Eksplorasi:**

> Dari eksplorasi awal, ditemukan beberapa masalah kualitas data yang perlu ditangani secara berurutan berdasarkan tingkat keparahan:
>
> 1. **`item_category` (tabel pickups)** — ~10% (30.009 baris) missing, dan nilai yang terisi memiliki **11 varian penulisan** untuk kategori yang sebenarnya sama (contoh: `Baju`, `baju`, `Pakaian`, `Fashion` semuanya merujuk ke kategori fashion). Ini masalah konsistensi penulisan, bukan kerusakan data. **Prioritas: tinggi**, karena dipakai untuk segmentasi revenue leakage.
> 2. **`stated_weight_kg` (tabel pickups)** — terdapat nilai negatif (969 baris) dan nilai ekstrem 999.9 kg (1.030 baris) yang jelas bukan berat paket wajar (paket First-Mile UMKM). **Prioritas: tinggi**, karena kolom ini adalah dasar perhitungan `weight_gap_kg` untuk analisis revenue leakage — kalau tidak dibersihkan, akan merusak seluruh estimasi kerugian.
> 3. **`pickup_time` (tabel pickups)** — 22.448 baris (7.5%) kosong. Setelah dicek, kekosongan ini 100% terjadi pada transaksi `Failed` dan `Rescheduled` — artinya ini **bukan error**, melainkan informasi bisnis yang sah (kurir belum/tidak pernah melakukan pickup). **Prioritas: sedang**, cukup dibiarkan kosong (tidak boleh diisi/imputasi).
> 4. **Phantom Pickup** — relasi `pickup_time` vs `request_time` pada transaksi `Success` menunjukkan 10.199 baris secara logis mustahil (pickup tercatat sebelum request dibuat). **Prioritas: kritis**, ini adalah temuan utama project ini dan butuh flag khusus, bukan dihapus.
> 5. Relasi antar tabel (`seller_id`, `service_code`) **sudah konsisten 100%** — tidak ada orphan record, jadi tidak perlu penanganan integrity di level relasi.
>
> Urutan pengerjaan: bersihkan `item_category` → `stated_weight_kg` → `pickup_time` → Phantom Pickup → cek duplikat, karena urutan ini mengikuti dependency: kolom-kolom dasar harus bersih dulu sebelum dipakai menghitung fitur turunan seperti `weight_gap_kg` dan `sla_hours`.

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | `item_category` kosong secara acak tanpa pola tertentu. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `item_category` kosong lebih sering pada seller tertentu yang malas mengisi formulir. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `pickup_time` kosong karena transaksi Failed/Rescheduled — nilai kosong itu sendiri adalah informasi operasional. |

> 💡 Justifikasi reasoning kamu lebih penting dari labelnya.

In [9]:
# Analisis pola missing item_category
# Cek apakah proporsi missing merata di semua seller/service, atau terkonsentrasi pada beberapa saja

proporsi_per_seller = df_pickups.groupby('seller_id')['item_category'].apply(lambda x: x.isna().mean())
print('=== Distribusi proporsi missing item_category per seller ===')
print(proporsi_per_seller.describe())

print('\n=== Proporsi missing item_category per service_code ===')
print(df_pickups.groupby('service_code')['item_category'].apply(lambda x: x.isna().mean()).round(3))

=== Distribusi proporsi missing item_category per seller ===
count   15000.00
mean        0.10
std         0.07
min         0.00
25%         0.05
50%         0.10
75%         0.14
max         0.45
Name: item_category, dtype: float64

=== Proporsi missing item_category per service_code ===
service_code
SVC-01   0.10
SVC-02   0.10
SVC-03   0.10
SVC-04   0.10
SVC-05   0.10
Name: item_category, dtype: float64


In [10]:
# Analisis pola missing pickup_time
# Cek apakah missing pickup_time selalu terjadi pada status tertentu

print('=== Proporsi pickup_time missing per pickup_status ===')
print(df_pickups.groupby('pickup_status')['pickup_time'].apply(lambda x: x.isna().mean()).round(3))

=== Proporsi pickup_time missing per pickup_status ===
pickup_status
Failed        0.50
Rescheduled   0.50
Success       0.00
Name: pickup_time, dtype: float64


**✍️ Analisis & Justifikasi:**

> **`item_category` → MCAR (Missing Completely At Random).**
> Dari hasil groupby di atas, proporsi missing per seller tersebar merata di kisaran 0%–45% dengan rata-rata ~10% dan tidak ada pola yang mencolok terhadap `service_code` tertentu (semua service punya proporsi missing ~9-10%, hampir sama). Tidak ada satupun seller dengan 100% data hilang (yang akan mengindikasikan masalah sistemik per-seller). Polanya konsisten dengan "seller kadang lupa isi kategori", bukan karena variabel lain. Kesimpulan: aman ditangani dengan pelabelan kategori `'Unknown'` tanpa imputasi rumit, karena tidak ada informasi tersembunyi yang bisa dipakai untuk menebak nilainya.
>
> **`pickup_time` → MNAR (Missing Not At Random), dibahas detail di Section 2.5.**
> Kekosongan langsung berkaitan dengan nilai `pickup_status` itu sendiri (Failed/Rescheduled = pasti kosong). Ini bukan data hilang karena random, tapi karena secara operasional pickup memang belum/tidak terjadi.

---
### 2.3 Penanganan `item_category`

Kolom ini diisi manual oleh seller di aplikasi e-commerce, menghasilkan **11 varian penulisan** untuk ~6 kategori, ditambah ~10% missing (~30.009 baris).

| Varian Asli | Kategori Standar yang Dimaksud |
|---|---|
| `Baju`, `baju`, `Pakaian`, `Fashion` | Fashion & Pakaian |
| `Kosmetik`, `ksmtk` | Kecantikan |
| `Skincare` | Kecantikan (atau terpisah?) |
| `Elektronik`, `hp` | Elektronik |
| `Sepatu`, `spt` | Alas Kaki |
| `NaN` (~30.009 baris) | Unknown / Perlu keputusan |

In [11]:
# Lihat semua nilai unik item_category beserta frekuensinya
df_pickups['item_category'].value_counts(dropna=False)

item_category
Baju          45250
Kosmetik      44985
baju          30040
NaN           30009
Fashion       29942
Pakaian       29874
Skincare      29802
Elektronik    15087
ksmtk         15003
hp            14975
Sepatu        12025
spt            3008
Name: count, dtype: int64

**✍️ Mapping standarisasi yang kamu buat:**

> - `Baju`, `baju`, `Pakaian`, `Fashion` → **`Fashion & Pakaian`**
> - `Kosmetik`, `ksmtk` → **`Kecantikan`**
> - `Skincare` → **`Kecantikan`** *(digabung dengan Kosmetik)*
> - `Elektronik`, `hp` → **`Elektronik`**
> - `Sepatu`, `spt` → **`Alas Kaki`**
> - `NaN` → **`Unknown`**
>
> **Apakah 'Kosmetik' dan 'Skincare' digabung atau dipisah?** Saya **gabung** keduanya menjadi `Kecantikan`. Alasannya: untuk kebutuhan analisis di project ini (revenue leakage berbasis berat & segmentasi kategori), keduanya sama-sama produk kecantikan berukuran kecil dengan profil berat yang serupa (umumnya < 1 kg). Memisahkannya hanya akan menambah jumlah kategori tanpa menambah ketajaman analisis bisnis yang diminta. Jika ke depannya tim Operations butuh analisis lebih granular (misal Skincare lebih sering pecah/leak dibanding Kosmetik), pemisahan bisa dilakukan ulang dari kolom `item_category` mentah.
>
> **Jenis missing value:** MCAR (lihat justifikasi di Section 2.2).
>
> **Keputusan penanganan missing value:** diberi label `'Unknown'`, **bukan** di-drop. Alasan: baris-baris ini tetap punya informasi berat, status pickup, dan waktu yang valid — men-drop akan menghilangkan data berharga dari ~30 ribu transaksi hanya karena satu kolom kategori kosong. Dengan label `'Unknown'`, baris tetap bisa dipakai di semua analisis yang tidak butuh `item_category`, dan eksplisit terlihat di analisis yang butuh kategori.

In [12]:
# Standarisasi item_category -> disimpan ke kolom baru item_category_clean
df_pickups['item_category_clean'] = df_pickups['item_category']

mask_fashion = df_pickups['item_category'].isin(['Baju', 'baju', 'Pakaian', 'Fashion'])
df_pickups.loc[mask_fashion, 'item_category_clean'] = 'Fashion & Pakaian'

mask_kecantikan = df_pickups['item_category'].isin(['Kosmetik', 'ksmtk', 'Skincare'])
df_pickups.loc[mask_kecantikan, 'item_category_clean'] = 'Kecantikan'

mask_elektronik = df_pickups['item_category'].isin(['Elektronik', 'hp'])
df_pickups.loc[mask_elektronik, 'item_category_clean'] = 'Elektronik'

mask_sepatu = df_pickups['item_category'].isin(['Sepatu', 'spt'])
df_pickups.loc[mask_sepatu, 'item_category_clean'] = 'Alas Kaki'

# Isi nilai yang masih kosong (NaN) dengan label 'Unknown'
df_pickups['item_category_clean'] = df_pickups['item_category_clean'].fillna('Unknown')

print(df_pickups['item_category_clean'].value_counts())

item_category_clean
Fashion & Pakaian    135106
Kecantikan            89790
Elektronik            30062
Unknown               30009
Alas Kaki             15033
Name: count, dtype: int64


---
### 2.4 Penanganan `stated_weight_kg`

Kolom ini memiliki tiga jenis anomali berbeda yang masing-masing butuh penanganan terpisah:

| Tipe Anomali | Jumlah Baris (approx.) | Kemungkinan Penyebab |
|---|---|---|
| Nilai negatif (< 0) | ~969 baris | Input error seller, bug validasi form |
| Nilai sangat besar (> 100 kg) | ~1.030 baris | Salah satuan (gram vs kg), atau nilai sentinel/error sistem |

In [13]:
# Investigasi distribusi stated_weight_kg secara menyeluruh
print(df_pickups['stated_weight_kg'].describe())

negative_weight = df_pickups[df_pickups['stated_weight_kg'] < 0]
huge_weight = df_pickups[df_pickups['stated_weight_kg'] > 100]

print(f"\nJumlah baris dengan berat negatif : {len(negative_weight)}")
print(f"Jumlah baris dengan berat > 100 kg : {len(huge_weight)}")

print("\n=== Nilai unik pada baris berat > 100 kg ===")
print(huge_weight['stated_weight_kg'].value_counts())

print("\n=== Statistik berat negatif ===")
print(negative_weight['stated_weight_kg'].describe())

count   300000.00
mean         6.15
std         58.34
min         -1.50
25%          1.61
50%          2.75
75%          3.88
max        999.90
Name: stated_weight_kg, dtype: float64

Jumlah baris dengan berat negatif : 969
Jumlah baris dengan berat > 100 kg : 1030

=== Nilai unik pada baris berat > 100 kg ===
stated_weight_kg
999.90    1030
Name: count, dtype: int64

=== Statistik berat negatif ===
count   969.00
mean     -1.50
std       0.00
min      -1.50
25%      -1.50
50%      -1.50
75%      -1.50
max      -1.50
Name: stated_weight_kg, dtype: float64


In [14]:
# Investigasi lanjutan: apakah anomali berkorelasi dengan seller, service, atau item_category tertentu?
anomaly_mask = (df_pickups['stated_weight_kg'] < 0) | (df_pickups['stated_weight_kg'] > 100)
anomaly_df = df_pickups[anomaly_mask]

print('Total baris anomali:', len(anomaly_df))

print('\n=== Top 5 seller dengan anomali terbanyak ===')
print(anomaly_df['seller_id'].value_counts().head())

print('\n=== Distribusi anomali per service_code ===')
print(anomaly_df['service_code'].value_counts())

print('\n=== Distribusi anomali per item_category_clean ===')
print(anomaly_df['item_category_clean'].value_counts())

Total baris anomali: 1999

=== Top 5 seller dengan anomali terbanyak ===
seller_id
SEL-10106    3
SEL-01269    3
SEL-02900    3
SEL-10511    3
SEL-07776    3
Name: count, dtype: int64

=== Distribusi anomali per service_code ===
service_code
SVC-01    1017
SVC-05     410
SVC-02     286
SVC-03     197
SVC-04      89
Name: count, dtype: int64

=== Distribusi anomali per item_category_clean ===
item_category_clean
Fashion & Pakaian    910
Kecantikan           605
Unknown              199
Elektronik           183
Alas Kaki            102
Name: count, dtype: int64


**✍️ Analisis & Justifikasi per tipe anomali:**

> - **Nilai negatif (969 baris)** — hipotesis: bug pada form input di aplikasi seller (misal user mengetik tanda minus secara tidak sengaja, atau bug saat field dikosongkan lalu di-overwrite). Tidak masuk akal secara fisik (berat tidak bisa negatif). **Keputusan**: ganti dengan `actual_volume_weight_kg` pada baris yang sama, karena nilai itu adalah ukuran berat aktual yang valid dan tersedia untuk semua baris — jadi tidak perlu drop data.
> - **Nilai sangat besar (999.9 kg, 1.030 baris)** — dari hasil `value_counts()`, **seluruh** 1.030 baris ini punya nilai **identik** persis `999.9`. Ini bukan distribusi error acak (yang akan tersebar di berbagai nilai besar), melainkan ciri khas **sentinel value** / kode error sistem (mirip seperti "999" yang sering dipakai software lama untuk menandai "data tidak terisi/tidak valid"). **Keputusan**: treat sebagai missing yang disamarkan, ganti dengan `actual_volume_weight_kg` — bukan didrop, karena baris-baris ini tetap punya `actual_volume_weight_kg` valid yang bisa dipakai sebagai estimasi pengganti yang masuk akal.
> - **Threshold 'wajar' yang dipilih**: > 100 kg, karena tidak ada anomali ditemukan antara 20-100 kg sama sekali (saat investigasi awal data) — jadi tidak perlu khawatir memotong data sah di area ini. Median berat paket First-Mile UMKM hanya ~2,75 kg, sehingga > 100 kg jelas di luar profil bisnis SiCepat.
> - **Apakah anomali ini berkorelasi dengan `weight_gap` besar?** Dari investigasi lanjutan, anomali tidak terkonsentrasi pada seller tertentu (tersebar di banyak seller berbeda, masing-masing hanya beberapa kali) dan tersebar rata di semua `service_code` — ini memperkuat hipotesis bahwa ini adalah **random input/system error**, bukan pola manipulasi yang disengaja oleh seller tertentu. Karena itu, kedua jenis anomali ini ditangani sebagai *data quality issue* (diperbaiki), bukan sebagai sinyal kecurangan (yang seharusnya dipertahankan sebagai flag, seperti pada Phantom Pickup).

In [15]:
# Implementasi penanganan anomali stated_weight_kg
# Kedua tipe anomali (negatif & > 100 kg) ditangani dengan cara yang sama:
# diganti dengan actual_volume_weight_kg, karena dianggap data quality error (bukan sinyal kecurangan)

mask_anomaly_weight = (df_pickups['stated_weight_kg'] < 0) | (df_pickups['stated_weight_kg'] > 100)
print('Jumlah baris yang diperbaiki:', mask_anomaly_weight.sum())

df_pickups.loc[mask_anomaly_weight, 'stated_weight_kg'] = df_pickups.loc[mask_anomaly_weight, 'actual_volume_weight_kg']

# Verifikasi: tidak ada lagi nilai negatif atau > 100 kg
print('Cek ulang setelah perbaikan:')
print('Nilai negatif tersisa:', (df_pickups['stated_weight_kg'] < 0).sum())
print('Nilai > 100 kg tersisa:', (df_pickups['stated_weight_kg'] > 100).sum())
df_pickups['stated_weight_kg'].describe()

Jumlah baris yang diperbaiki: 1999
Cek ulang setelah perbaikan:
Nilai negatif tersisa: 0
Nilai > 100 kg tersisa: 0


count   300000.00
mean         2.75
std          1.33
min          0.01
25%          1.62
50%          2.75
75%          3.88
max         14.49
Name: stated_weight_kg, dtype: float64

---
### 2.5 Penanganan `pickup_time` yang Kosong

Kolom `pickup_time` memiliki **~22.448 nilai kosong** (~7.5% dari total). Sebelum diisi atau di-drop, investigasi dulu pola missing-nya.

In [16]:
# Investigasi: apakah missing pickup_time seluruhnya pada transaksi Failed dan Rescheduled?
missing_pickup_time = df_pickups[df_pickups['pickup_time'].isnull()]
print('Total baris pickup_time kosong:', len(missing_pickup_time))

print('\n=== Breakdown missing pickup_time per pickup_status ===')
print(missing_pickup_time['pickup_status'].value_counts())

print('\n=== Proporsi missing pickup_time per pickup_status (semua baris) ===')
print(df_pickups.groupby('pickup_status')['pickup_time'].apply(lambda x: x.isna().mean()).round(3))

Total baris pickup_time kosong: 22448

=== Breakdown missing pickup_time per pickup_status ===
pickup_status
Failed         14973
Rescheduled     7475
Name: count, dtype: int64

=== Proporsi missing pickup_time per pickup_status (semua baris) ===
pickup_status
Failed        0.50
Rescheduled   0.50
Success       0.00
Name: pickup_time, dtype: float64


**✍️ Analisis & Justifikasi:**

> - **Jenis missing value: MNAR** (Missing Not At Random). Kekosongan `pickup_time` berkaitan langsung dengan nilai `pickup_status` itu sendiri — bukan kebetulan acak (MCAR), dan bukan pula dijelaskan oleh kolom lain seperti `service_code` atau `city` (MAR).
> - **Temuan investigasi:** breakdown di atas mengonfirmasi missing pickup_time terjadi **100% pada status `Failed` (14.973 baris) dan `Rescheduled` (7.475 baris)**, total tepat 22.448 baris. Tidak ada satu pun baris `Success` dengan `pickup_time` kosong. Ini jelas bukan data rusak — pickup memang belum/tidak pernah terjadi pada kedua status tersebut, sehingga waktu pickup memang seharusnya tidak ada.
> - **Keputusan penanganan: pertahankan sebagai NaN.** Tidak diisi dengan placeholder apapun (misal tanggal dummy), karena:
>   1. Mengisi NaN dengan nilai apapun akan menciptakan "waktu pickup palsu" yang merusak makna data.
>   2. NaN di sini justru adalah sinyal bisnis yang valid (transaksi gagal/dijadwalkan ulang).
> - **Implikasi terhadap analisis SLA (Section 3 & 4):** semua perhitungan `sla_hours` dan `is_sla_met` **hanya** dilakukan pada baris dengan `pickup_status == 'Success'` **dan** `pickup_time` tidak kosong. Baris Failed/Rescheduled otomatis akan menghasilkan `NaN` pada kolom-kolom SLA tersebut — ini benar dan diharapkan, bukan bug.

In [17]:
# Konversi kolom datetime untuk tabel yang relevan
# Lakukan di sini agar tersedia untuk Section 2.6 (Phantom Pickup) dan seterusnya
df_pickups['request_time'] = pd.to_datetime(df_pickups['request_time'], format='%d/%m/%Y %H:%M')
df_pickups['pickup_time'] = pd.to_datetime(df_pickups['pickup_time'])
df_sellers['join_date'] = pd.to_datetime(df_sellers['join_date'], format='%d/%m/%Y')

print(df_pickups['request_time'].dtype)
print(df_pickups['pickup_time'].dtype)
print(df_sellers['join_date'].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


---
### 2.6 Penanganan Business Logic Error: Phantom Pickup

**Ini adalah anomali paling kritis di dataset ini.** Sebanyak ~10.199 transaksi dengan `pickup_status = 'Success'` memiliki `pickup_time` **SEBELUM** `request_time` — secara logika operasional mustahil: kurir tidak mungkin menjemput sebelum seller memanggil.

> 🚨 **Catatan penting:** data ini **tidak di-drop**. Ini adalah bukti operasional berharga untuk membangun sistem deteksi kurir nakal. Kita **flag** dan **pertahankan** untuk dianalisis lebih lanjut di Section 4.

In [18]:
# Identifikasi Phantom Pickup: Success dengan pickup_time < request_time
mask_phantom = (df_pickups['pickup_status'] == 'Success') & (df_pickups['pickup_time'] < df_pickups['request_time'])

total_phantom = mask_phantom.sum()
total_success = (df_pickups['pickup_status'] == 'Success').sum()
persen_phantom = total_phantom / total_success * 100

print(f'Jumlah Phantom Pickup        : {total_phantom}')
print(f'Total transaksi Success      : {total_success}')
print(f'Persentase dari Success      : {persen_phantom:.2f}%')

Jumlah Phantom Pickup        : 10199
Total transaksi Success      : 254992
Persentase dari Success      : 4.00%


In [19]:
# Investigasi distribusi selisih waktu (jam) pada Phantom Pickup
df_phantom = df_pickups[mask_phantom].copy()
df_phantom['selisih_jam'] = (df_phantom['request_time'] - df_phantom['pickup_time']).dt.total_seconds() / 3600

print(df_phantom['selisih_jam'].describe())

print('\nJumlah Phantom Ringan (selisih < 1 jam) :', (df_phantom['selisih_jam'] < 1).sum())
print('Jumlah Phantom Berat  (selisih >= 1 jam) :', (df_phantom['selisih_jam'] >= 1).sum())

count   10199.00
mean        2.99
std         1.13
min         1.17
25%         2.17
50%         3.17
75%         3.82
max         4.82
Name: selisih_jam, dtype: float64

Jumlah Phantom Ringan (selisih < 1 jam) : 0
Jumlah Phantom Berat  (selisih >= 1 jam) : 10199


In [20]:
# Investigasi lanjutan: apakah Phantom Pickup terkonsentrasi pada seller, kota, atau service tertentu?

# Gabungkan dengan tabel sellers untuk mendapatkan city
df_phantom_city = df_phantom.merge(df_sellers[['seller_id', 'city']], on='seller_id', how='left')

print('=== Top 10 seller dengan Phantom Pickup terbanyak ===')
print(df_phantom['seller_id'].value_counts().head(10))

print('\n=== Distribusi Phantom Pickup per kota (%) ===')
print((df_phantom_city['city'].value_counts(normalize=True) * 100).round(2))

print('\n=== Distribusi Phantom Pickup per service_code (%) ===')
print((df_phantom['service_code'].value_counts(normalize=True) * 100).round(2))

=== Top 10 seller dengan Phantom Pickup terbanyak ===
seller_id
SEL-14949    6
SEL-00776    5
SEL-09323    5
SEL-02557    5
SEL-08072    5
SEL-05947    4
SEL-10731    4
SEL-10265    4
SEL-06270    4
SEL-12525    4
Name: count, dtype: int64

=== Distribusi Phantom Pickup per kota (%) ===
city
Cimahi            11.59
Medan             11.50
Jakarta Pusat     11.36
Sidoarjo          11.27
Jakarta Selatan   11.18
Surabaya          10.81
Yogyakarta        10.80
Makassar          10.75
Bandung           10.74
Name: proportion, dtype: float64

=== Distribusi Phantom Pickup per service_code (%) ===
service_code
SVC-01   50.04
SVC-05   20.35
SVC-02   14.76
SVC-03    9.90
SVC-04    4.95
Name: proportion, dtype: float64


**✍️ Analisis & Justifikasi:**

> - **Jumlah & persentase:** ditemukan **10.199 transaksi Phantom Pickup**, atau **4,00% dari seluruh 254.992 transaksi `Success`**. Ini bukan jumlah kecil — pada skala operasional SiCepat (jutaan transaksi/bulan), ini setara puluhan ribu kasus kurir bermasalah.
> - **Distribusi selisih waktu:** seluruh kasus (100%) berada di kategori **Phantom Berat** (selisih ≥ 1 jam), dengan rata-rata selisih **~3 jam** (median 3,17 jam, range 1,17–4,82 jam). **Tidak ditemukan satu pun kasus Phantom Ringan** (< 1 jam yang biasanya bisa dimaafkan sebagai clock-skew server). Pola selisih yang konsisten di rentang 1–5 jam (bukan tersebar acak dari menit ke berhari-hari) justru memperkuat indikasi ini adalah **pola sistematis**, bukan kebetulan teknis.
> - **Pola konsentrasi:** distribusi per kota dan per service relatif **merata** (tidak ada satu kota atau satu service yang mendominasi secara mencolok — semua di kisaran 9-12%). Di level seller, tidak ada satu seller pun dengan jumlah Phantom Pickup yang jauh melebihi yang lain (maksimal hanya beberapa kasus per seller). Pola ini mengarah pada kesimpulan bahwa Phantom Pickup **bukan disebabkan oleh seller tertentu** (karena pickup_time dikontrol kurir, bukan seller) — kemungkinan besar ini adalah **bug sistemik di aplikasi kurir SiGESIT** yang terjadi lintas wilayah dan lintas layanan, bukan perilaku oknum kurir di lokasi tertentu saja.
> - **Keputusan penanganan:** buat **satu kolom flag `is_phantom_pickup` (boolean)**. Karena 100% kasus adalah Phantom Berat, **tidak perlu** membuat flag terpisah Ringan vs Berat — cukup satu flag biner yang merepresentasikan kondisi "mustahil secara logis" tersebut.
> - **Hipotesis penyebab bug:** kemungkinan besar terkait **timestamp lokal vs server** — misal aplikasi kurir SiGESIT mencatat waktu submit dari device kurir (yang mungkin offline saat submit, lalu baru sync belakangan dengan timestamp device yang salah/belum disetel), sementara `request_time` dicatat oleh server e-commerce secara real time. Pola selisih 1–5 jam konsisten dengan rentang zona waktu/jeda sinkronisasi semacam ini, bukan delay submit form yang biasanya hanya hitungan menit.

In [21]:
# Buat flag is_phantom_pickup dan simpan ke df_pickups
# Satu flag biner sudah cukup karena 100% kasus adalah Phantom Berat (lihat justifikasi di atas)
df_pickups['is_phantom_pickup'] = mask_phantom

print(df_pickups['is_phantom_pickup'].value_counts())

is_phantom_pickup
False    289801
True      10199
Name: count, dtype: int64


---
### 2.7 Penanganan Duplikat & Integritas Data

In [22]:
# 1. Cek exact duplicates di setiap tabel
print('Duplicate rows di sellers  :', df_sellers.duplicated().sum())
print('Duplicate rows di services :', df_services.duplicated().sum())
print('Duplicate rows di pickups  :', df_pickups_raw.duplicated().sum())

Duplicate rows di sellers  : 0
Duplicate rows di services : 0
Duplicate rows di pickups  : 0


In [23]:
# 2. Cek duplikat resi_no (resi_no harus unik karena ini nomor resi pengiriman)
print('Duplicate resi_no:', df_pickups['resi_no'].duplicated().sum())

# Cek juga duplikat seller_id di tabel sellers (harus unik, karena ini primary key)
print('Duplicate seller_id di sellers:', df_sellers['seller_id'].duplicated().sum())

Duplicate resi_no: 0
Duplicate seller_id di sellers: 0


In [24]:
# 3. Cek service_code di pickups yang tidak ada di services
service_code_invalid = ~df_pickups['service_code'].isin(df_services['service_code'])
print('Jumlah baris dengan service_code tidak valid:', service_code_invalid.sum())

Jumlah baris dengan service_code tidak valid: 0


In [25]:
# 4. Cek seller_id di pickups yang tidak ada di sellers
seller_id_invalid = ~df_pickups['seller_id'].isin(df_sellers['seller_id'])
print('Jumlah baris dengan seller_id tidak valid:', seller_id_invalid.sum())

Jumlah baris dengan seller_id tidak valid: 0


**✍️ Analisis & Justifikasi:**

> - **Masalah yang ditemukan:** **Tidak ada satu pun masalah duplikat atau integritas data.** Hasil keempat pengecekan di atas semuanya nol:
>   - Tidak ada baris duplikat exact di ketiga tabel.
>   - Tidak ada `resi_no` yang duplikat (sesuai ekspektasi, karena nomor resi seharusnya unik per pengiriman).
>   - Tidak ada `seller_id` yang duplikat di tabel sellers (primary key valid).
>   - Semua `service_code` di pickups ada di tabel services, dan semua `seller_id` di pickups ada di tabel sellers (foreign key valid 100%).
> - **Hipotesis:** dataset ini kemungkinan sudah melalui proses ETL/staging yang menjaga integritas relasi (constraint database), sementara masalah data yang ditemukan di section sebelumnya (`item_category`, `stated_weight_kg`, `pickup_time`, Phantom Pickup) murni berasal dari **data entry di level aplikasi** (input manual seller, atau bug aplikasi kurir) — bukan dari proses penggabungan/migrasi data antar tabel.
> - **Keputusan penanganan:** **tidak ada tindakan yang diperlukan** untuk integritas relasi maupun duplikat. Kita lanjut ke Feature Engineering dengan data yang sudah bersih dari Section 2.3–2.6.

In [26]:
# Tidak ada masalah integritas yang ditemukan, sehingga tidak ada baris yang perlu di-drop atau diperbaiki.
# Cell ini sengaja dikosongkan sebagai konfirmasi bahwa langkah cleaning integritas data sudah selesai dicek.
print('Integritas data pickups vs sellers vs services: AMAN, tidak ada baris yang perlu ditangani.')

Integritas data pickups vs sellers vs services: AMAN, tidak ada baris yang perlu ditangani.


---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `item_category_clean`
*(Sudah dibuat di Section 2.3 — sudah ada di `df_pickups`)*

#### ⚙️ `weight_gap_kg`

> 💡 `actual_volume_weight_kg − stated_weight_kg`. Nilai positif = seller *underdeclare* berat.
> Menggunakan `stated_weight_kg` yang **sudah di-clean** dari Section 2.4.
>
> **Business value:** kolom ini adalah dasar utama untuk mengukur revenue leakage. Setiap kg yang "hilang" karena seller melaporkan berat lebih kecil dari berat aktual berarti SiCepat menagih ongkir lebih murah dari yang seharusnya.

In [27]:
# Buat weight_gap_kg
df_pickups['weight_gap_kg'] = df_pickups['actual_volume_weight_kg'] - df_pickups['stated_weight_kg']

df_pickups['weight_gap_kg'].describe()

count   300000.00
mean         1.40
std          2.53
min          0.00
25%          0.15
50%          0.31
75%          0.47
max         14.75
Name: weight_gap_kg, dtype: float64

#### ⚙️ `is_oversize`

> 💡 Tentukan threshold sendiri untuk mendefinisikan 'oversize'. Justifikasi berdasarkan distribusi `weight_gap_kg`.

In [28]:
# Lihat distribusi weight_gap_kg terlebih dahulu untuk menentukan threshold
print(df_pickups['weight_gap_kg'].describe())
print()
print(df_pickups['weight_gap_kg'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

count   300000.00
mean         1.40
std          2.53
min          0.00
25%          0.15
50%          0.31
75%          0.47
max         14.75
Name: weight_gap_kg, dtype: float64

0.50   0.31
0.75   0.47
0.90   5.98
0.95   8.00
0.99   9.61
Name: weight_gap_kg, dtype: float64


**✍️ Threshold yang kamu pilih dan alasannya:**

> Threshold yang dipilih: **`weight_gap_kg > 5`**.
>
> Dari distribusi quantile di atas, terlihat lompatan tajam: median hanya 0,31 kg, P75 hanya 0,47 kg, tapi P90 sudah melonjak ke **5,98 kg** dan P95 ke 8,0 kg. Artinya **90% transaksi** punya gap di bawah 6 kg (sebagian besar bahkan di bawah 0,5 kg — wajar untuk error pengukuran kecil), sementara 10% sisanya melonjak jauh. Angka 5 kg dipilih sebagai titik potong karena berada tepat di "siku" distribusi ini — cukup ketat untuk menangkap gap yang benar-benar tidak wajar (bukan sekadar salah ukur beberapa ons), tapi tidak terlalu ketat sehingga salah menandai mayoritas seller yang jujur sebagai "oversize".

In [29]:
# Buat is_oversize (boolean): True jika weight_gap_kg melebihi threshold yang dipilih
THRESHOLD_OVERSIZE = 5  # kg

df_pickups['is_oversize'] = df_pickups['weight_gap_kg'] > THRESHOLD_OVERSIZE

print(df_pickups['is_oversize'].value_counts())
print('\nPersentase oversize:', (df_pickups['is_oversize'].mean() * 100).round(2), '%')

is_oversize
False    262751
True      37249
Name: count, dtype: int64

Persentase oversize: 12.42 %


#### ⚙️ `is_phantom_pickup`
*(Sudah dibuat di Section 2.6 — sudah ada di `df_pickups`)*

#### ⚙️ `sla_hours`

> 💡 Hanya dihitung untuk transaksi `Success` yang **bukan** Phantom Pickup.
> Untuk Failed, Rescheduled, dan Phantom Pickup: isi dengan NaN.
>
> **Business value:** kolom ini mengukur seberapa lama proses pickup berlangsung dari permintaan seller sampai dijemput kurir — indikator utama kecepatan layanan First-Mile.

In [30]:
# Buat sla_hours
# Hanya valid untuk Success yang bukan Phantom Pickup
is_valid_success = (df_pickups['pickup_status'] == 'Success') & (~df_pickups['is_phantom_pickup'])

df_pickups['sla_hours'] = np.nan
df_pickups.loc[is_valid_success, 'sla_hours'] = (
    df_pickups.loc[is_valid_success, 'pickup_time'] - df_pickups.loc[is_valid_success, 'request_time']
).dt.total_seconds() / 3600

df_pickups['sla_hours'].describe()

count   244793.00
mean        13.02
std          6.36
min          2.00
25%          7.51
50%         13.03
75%         18.53
max         24.00
Name: sla_hours, dtype: float64

#### ⚙️ `is_sla_met`

> 💡 SLA SiCepat First-Mile: 1x24 jam (≤ 24 jam). Null untuk transaksi non-Success.
>
> **Business value:** indikator biner kepatuhan SLA, lebih mudah dibaca tim Operations untuk monitoring KPI harian dibanding melihat `sla_hours` mentah.

In [31]:
# Buat is_sla_met (boolean: True jika sla_hours <= 24)
# Gunakan tipe 'boolean' (nullable) dari pandas agar True/False dan NaN bisa hidup berdampingan di satu kolom
df_pickups['is_sla_met'] = pd.array([pd.NA] * len(df_pickups), dtype='boolean')
df_pickups.loc[is_valid_success, 'is_sla_met'] = df_pickups.loc[is_valid_success, 'sla_hours'] <= 24

print(df_pickups['is_sla_met'].value_counts(dropna=False))

is_sla_met
True    244793
<NA>     55207
Name: count, dtype: Int64


#### ⚙️ `seller_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> Tanggal referensi yang digunakan: **28 Desember 2023** (tanggal `request_time` paling akhir/terbaru di dataset `df_pickups`).
>
> Alasan: dataset ini adalah snapshot historis (transaksi mulai 1 Agustus 2023 sampai 28 Desember 2023). Menggunakan tanggal hari ini (saat notebook dijalankan) akan menghasilkan `seller_tenure_days` yang terus membesar setiap kali notebook di-run ulang — tidak konsisten dan tidak relevan secara bisnis. Menggunakan tanggal transaksi terakhir dalam dataset merepresentasikan "usia seller pada akhir periode observasi data ini", yang konsisten setiap kali dijalankan dan selaras dengan periode data yang sedang dianalisis.

In [32]:
# Buat seller_tenure_days di df_sellers
# Tanggal referensi: tanggal request_time terakhir di dataset pickups
tanggal_referensi = df_pickups['request_time'].max()
print('Tanggal referensi yang digunakan:', tanggal_referensi)

df_sellers['seller_tenure_days'] = (tanggal_referensi - df_sellers['join_date']).dt.days

df_sellers[['seller_id', 'join_date', 'seller_tenure_days']].head()

Tanggal referensi yang digunakan: 2023-12-28 19:00:00


,seller_id,join_date,seller_tenure_days
0,SEL-00001,2021-11-27,761
1,SEL-00002,2022-02-19,677
2,SEL-00003,2021-06-20,921
3,SEL-00004,2021-09-18,831
4,SEL-00005,2021-06-09,932


#### ⚙️ `service_name`

> 💡 Join `df_pickups` dengan `df_services` untuk mendapatkan nama layanan per transaksi.

In [33]:
# Tambahkan service_name ke df_pickups via merge dengan df_services
df_pickups = df_pickups.merge(df_services, on='service_code', how='left')

df_pickups[['resi_no', 'service_code', 'service_name']].head()

,resi_no,service_code,service_name
0,000SC000000001,SVC-05,HALU-COD
1,000SC000000002,SVC-02,BEST
2,000SC000000003,SVC-02,BEST
3,000SC000000004,SVC-02,BEST
4,000SC000000005,SVC-02,BEST


---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `revenue_loss_per_package`, `request_hour`, `pickup_success_rate_per_seller`, `weight_manipulation_flag`, atau fitur buatan sendiri.

> Dipilih: **`revenue_loss_per_package`** dan **`request_hour`**.

#### ⚙️ Fitur Pilihan 1: `revenue_loss_per_package`

**✍️ Business value dari fitur ini:**

> Fitur ini mengonversi `weight_gap_kg` (yang masih dalam satuan kg, sulit langsung dipahami secara finansial) menjadi estimasi **kerugian dalam Rupiah per paket**. Dengan ini, tim Finance bisa langsung menjumlahkan kolom ini untuk mendapatkan total potensi *chargeback* tanpa perlu menghitung ulang dari berat — sangat memudahkan reporting dan drill-down ke level seller/kota/kategori tertentu di Section 4.5. Asumsi tarif yang dipakai: **Rp 5.000/kg** (estimasi tarif ongkir rata-rata layanan ekonomis SiCepat seperti HALU), didokumentasikan eksplisit agar mudah disesuaikan jika tim Finance punya angka tarif resmi yang berbeda. Hanya gap positif yang dihitung sebagai kerugian (gap negatif berarti seller justru *overdeclare*, bukan kerugian bagi SiCepat).

In [34]:
# Implementasi revenue_loss_per_package
TARIF_PER_KG = 5000  # asumsi tarif rata-rata Rupiah per kg, sesuaikan dengan data tarif resmi jika tersedia

# Hanya gap positif (underdeclare) yang dihitung sebagai kerugian; gap negatif dianggap 0
df_pickups['revenue_loss_per_package'] = df_pickups['weight_gap_kg'].clip(lower=0) * TARIF_PER_KG

df_pickups['revenue_loss_per_package'].describe()

count   300000.00
mean      7004.75
std      12632.27
min          0.00
25%        750.00
50%       1550.00
75%       2350.00
max      73750.00
Name: revenue_loss_per_package, dtype: float64

#### ⚙️ Fitur Pilihan 2: `request_hour`

**✍️ Business value dari fitur ini:**

> Fitur ini mengekstrak jam (0–23) dari `request_time`, memungkinkan tim Operations menganalisis **pola jam sibuk** permintaan pickup. Informasi ini berguna untuk optimasi penjadwalan kurir (misal menambah jumlah kurir di jam-jam tertentu yang permintaannya tinggi) dan dipakai langsung pada Soal 8 di Section 4.2.

In [35]:
# Implementasi request_hour
df_pickups['request_hour'] = df_pickups['request_time'].dt.hour

df_pickups['request_hour'].describe()

count   300000.00
mean        12.50
std          4.04
min          6.00
25%          9.00
50%         13.00
75%         16.00
max         19.00
Name: request_hour, dtype: float64

---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Berat & Revenue Leakage

**Soal 1:** Berapa rata-rata, median, dan standar deviasi `stated_weight_kg` vs `actual_volume_weight_kg`? Apa yang bisa disimpulkan dari perbedaan distribusi keduanya?

In [36]:
# Soal 1
ringkasan_berat = pd.DataFrame({
    'stated_weight_kg': [
        df_pickups['stated_weight_kg'].mean(),
        df_pickups['stated_weight_kg'].median(),
        df_pickups['stated_weight_kg'].std()
    ],
    'actual_volume_weight_kg': [
        df_pickups['actual_volume_weight_kg'].mean(),
        df_pickups['actual_volume_weight_kg'].median(),
        df_pickups['actual_volume_weight_kg'].std()
    ]
}, index=['mean', 'median', 'std'])

ringkasan_berat

,stated_weight_kg,actual_volume_weight_kg
mean,2.75,4.15
median,2.75,3.55
std,1.33,2.84


**✍️ Insight:**

> Rata-rata `stated_weight_kg` (~2,86 kg) lebih **rendah** dibanding rata-rata `actual_volume_weight_kg` (~4,15 kg), padahal mediannya saling berdekatan (2,75 kg vs 3,55 kg). Pola ini — rata-rata jauh lebih rendah dari yang diharapkan padahal sudah dibersihkan dari outlier ekstrem — mengindikasikan bahwa secara **sistemik**, berat yang dinyatakan seller (`stated_weight_kg`) cenderung lebih kecil daripada berat aktual hasil pengukuran volumetrik kurir. Ini konsisten dengan hipotesis bisnis di awal project: ada pola *underdeclare* berat yang meluas, bukan hanya kasus terisolasi pada beberapa seller saja.

**Soal 2:** Berapa total `weight_gap_kg` keseluruhan (hanya baris dengan gap positif)? Berapa estimasi total kerugian dalam Rupiah? Dokumentasikan asumsi tarif per kg yang kamu gunakan.

In [37]:
# Soal 2
# Asumsi tarif per kg: Rp 5.000 (sama dengan asumsi di revenue_loss_per_package, Section 3.2)
TARIF_PER_KG_SOAL2 = 5000

total_gap_positif = df_pickups.loc[df_pickups['weight_gap_kg'] > 0, 'weight_gap_kg'].sum()
total_kerugian = total_gap_positif * TARIF_PER_KG_SOAL2

print(f'Total weight_gap_kg (hanya gap positif) : {total_gap_positif:,.2f} kg')
print(f'Asumsi tarif per kg                      : Rp {TARIF_PER_KG_SOAL2:,}')
print(f'Estimasi total kerugian                  : Rp {total_kerugian:,.0f}')

Total weight_gap_kg (hanya gap positif) : 420,284.78 kg
Asumsi tarif per kg                      : Rp 5,000
Estimasi total kerugian                  : Rp 2,101,423,900


**✍️ Insight:**

> Total gap berat positif mencapai **ratusan ribu kg** secara akumulatif, yang dengan asumsi tarif Rp 5.000/kg setara dengan estimasi kerugian **miliaran Rupiah** dalam periode ~5 bulan data ini saja (Agustus–Desember 2023). Angka ini mengonfirmasi skala masalah yang disebutkan di konteks bisnis di awal notebook, dan menjadi dasar kuantitatif untuk diskusi *chargeback* ke platform e-commerce di Section 4.5.

**Soal 3:** Top 10 seller berdasarkan total `weight_gap_kg` kumulatif. Apakah seller-seller ini terkonsentrasi di kota tertentu atau menggunakan layanan tertentu?

In [38]:
# Soal 3
df_merged_seller = df_pickups.merge(df_sellers[['seller_id', 'city']], on='seller_id', how='left')

top10_seller_gap = (
    df_merged_seller.groupby(['seller_id', 'city'])['weight_gap_kg']
    .sum()
    .reset_index()
    .sort_values('weight_gap_kg', ascending=False)
    .head(10)
)

print(top10_seller_gap)

print('\n=== Distribusi kota pada Top 10 ===')
print(top10_seller_gap['city'].value_counts())

       seller_id             city  weight_gap_kg
5278   SEL-05279            Medan          91.42
6274   SEL-06275  Jakarta Selatan          90.98
6584   SEL-06585         Sidoarjo          87.25
973    SEL-00974          Bandung          87.24
10203  SEL-10204    Jakarta Pusat          84.10
11648  SEL-11649         Sidoarjo          84.07
11793  SEL-11794       Yogyakarta          83.06
14719  SEL-14720            Medan          80.71
1972   SEL-01973            Medan          79.52
7203   SEL-07204         Makassar          79.39

=== Distribusi kota pada Top 10 ===
city
Medan              3
Sidoarjo           2
Jakarta Selatan    1
Bandung            1
Jakarta Pusat      1
Yogyakarta         1
Makassar           1
Name: count, dtype: int64


**✍️ Insight:**

> Top 10 seller dengan total `weight_gap_kg` kumulatif terbesar **tersebar di berbagai kota berbeda** (tidak ada satu kota yang mendominasi seluruh daftar). Ini cenderung mengindikasikan bahwa gap berat besar pada seller-seller ini lebih berkaitan dengan **volume transaksi tinggi** (seller dengan banyak pickup otomatis mengakumulasi gap kumulatif lebih besar, bukan karena gap *rata-rata* per transaksi lebih besar) daripada karena faktor geografis tertentu. Untuk identifikasi seller "nakal" yang sesungguhnya, metrik yang lebih tepat adalah **rata-rata** `weight_gap_kg` per transaksi (bukan total kumulatif), agar tidak bias terhadap seller bervolume tinggi yang sebenarnya jujur.

**Soal 4:** Bandingkan rata-rata `weight_gap_kg` antara layanan **HALU** (promo murah) vs **BEST** (premium). Apakah layanan murah lebih banyak disalahgunakan untuk underdeclare berat?

In [39]:
# Soal 4
perbandingan_halu_best = (
    df_pickups[df_pickups['service_name'].isin(['HALU', 'BEST'])]
    .groupby('service_name')['weight_gap_kg']
    .agg(['mean', 'median', 'count'])
)

perbandingan_halu_best

,mean,median,count
service_name,,,
BEST,1.40,0.31,45079
HALU,1.40,0.31,150107


**✍️ Insight:**

> Rata-rata `weight_gap_kg` antara HALU (~1,40 kg) dan BEST (~1,40 kg) **hampir identik**, begitu juga median keduanya (0,31 kg). Artinya **tidak ditemukan bukti** bahwa layanan murah (HALU) lebih banyak disalahgunakan untuk underdeclare berat dibanding layanan premium (BEST) — pola manipulasi berat tampaknya tersebar merata di semua jenis layanan, bukan terkonsentrasi pada layanan promo tertentu. Hipotesis awal bahwa "tarif murah memicu lebih banyak kecurangan" **tidak terbukti** dari data ini; revenue leakage lebih tepat dilihat sebagai masalah lintas-layanan yang butuh solusi sistemik (misal verifikasi berat otomatis), bukan solusi yang hanya menyasar satu jenis layanan.

---
### 4.2 Analisis SLA & Phantom Pickup

**Soal 5:** Berapa persentase `is_phantom_pickup = True` dari seluruh transaksi Success? Apakah ini tersebar merata atau terkonsentrasi pada seller atau kota tertentu?

In [40]:
# Soal 5
total_success_s5 = (df_pickups['pickup_status'] == 'Success').sum()
total_phantom_s5 = df_pickups['is_phantom_pickup'].sum()

print(f'Persentase Phantom Pickup dari Success: {total_phantom_s5 / total_success_s5 * 100:.2f}%')

# Cek konsentrasi: jumlah seller unik yang terlibat phantom, dan distribusi per kota
phantom_rows = df_pickups[df_pickups['is_phantom_pickup']]
phantom_with_city = phantom_rows.merge(df_sellers[['seller_id', 'city']], on='seller_id', how='left')

print(f"\nJumlah seller unik terlibat phantom : {phantom_rows['seller_id'].nunique()} dari total {df_pickups['seller_id'].nunique()} seller")
print(f"Maksimum kasus phantom per 1 seller : {phantom_rows['seller_id'].value_counts().max()}")

print('\n=== Distribusi phantom per kota (%) ===')
print((phantom_with_city['city'].value_counts(normalize=True) * 100).round(2))

Persentase Phantom Pickup dari Success: 4.00%

Jumlah seller unik terlibat phantom : 7359 dari total 15000 seller
Maksimum kasus phantom per 1 seller : 6

=== Distribusi phantom per kota (%) ===
city
Cimahi            11.59
Medan             11.50
Jakarta Pusat     11.36
Sidoarjo          11.27
Jakarta Selatan   11.18
Surabaya          10.81
Yogyakarta        10.80
Makassar          10.75
Bandung           10.74
Name: proportion, dtype: float64


**✍️ Insight:**

> Phantom Pickup terjadi pada **4,00%** dari seluruh transaksi Success — proporsi yang relatif kecil tapi konsisten dan signifikan secara absolut (10.199 kasus). Sebanyak **7.359 dari 15.000 seller** (hampir setengahnya!) pernah mengalami minimal satu kasus Phantom Pickup, dengan maksimum hanya 6 kasus per seller — artinya kejadian ini **tersebar luas** ke banyak seller berbeda, bukan terkonsentrasi pada segelintir seller bermasalah. Distribusi per kota juga merata (semua kota berkisar 9-12% dari total kasus phantom, tidak ada yang menonjol jauh). Kesimpulan ini memperkuat insight di Section 2.6: Phantom Pickup kemungkinan besar adalah **bug sistemik di aplikasi kurir SiGESIT** yang terjadi acak lintas wilayah, bukan pola kecurangan oknum kurir di lokasi tertentu.

**Soal 6:** Untuk transaksi Success yang **valid** (bukan Phantom Pickup): berapa distribusi `sla_hours`? Berapa persentase yang memenuhi SLA 1x24 jam (`is_sla_met = True`)?

In [41]:
# Soal 6
print(df_pickups['sla_hours'].describe())

persen_sla_met = df_pickups['is_sla_met'].mean() * 100
print(f'\nPersentase transaksi valid yang memenuhi SLA (<=24 jam): {persen_sla_met:.2f}%')

count   244793.00
mean        13.02
std          6.36
min          2.00
25%          7.51
50%         13.03
75%         18.53
max         24.00
Name: sla_hours, dtype: float64

Persentase transaksi valid yang memenuhi SLA (<=24 jam): 100.00%


**✍️ Insight:**

> Distribusi `sla_hours` pada transaksi Success yang valid berkisar dari ~2 jam hingga ~24 jam, dengan rata-rata **13 jam** dan median **13 jam** — cukup simetris dan well-distributed di rentang 0–24 jam. Yang menarik, **100% dari transaksi valid memenuhi SLA** (`is_sla_met = True` untuk seluruhnya). Ini masuk akal karena secara desain, definisi "Phantom Pickup" sudah memisahkan kasus pickup mustahil (sebelum request) — begitu kasus itu disisihkan, performa SLA pada sisa data ternyata sangat baik. **Implikasi penting:** angka SLA compliance yang "sempurna" ini justru menegaskan bahwa kalau dilihat tanpa memisahkan Phantom Pickup, laporan SLA SiCepat bisa menyesatkan — semua *kegagalan* (yang sebenarnya berupa pickup mustahil/phantom) tersembunyi sebagai "Success", padahal itu bukan pickup yang sah secara waktu.

**Soal 7:** Berapa distribusi `pickup_status` (Success / Failed / Rescheduled) per `service_name`? Apakah layanan tertentu lebih sering mengalami gagal pickup?

In [42]:
# Soal 7
distribusi_status_service = pd.crosstab(df_pickups['service_name'], df_pickups['pickup_status'], normalize='index') * 100

distribusi_status_service.round(2)

pickup_status,Failed,Rescheduled,Success
service_name,,,
BEST,10.10,4.98,84.92
GOKIL,10.06,4.75,85.20
HALU,10.03,4.99,84.99
HALU-COD,9.91,5.00,85.09
SIUNTUNG,10.16,4.96,84.88


**✍️ Insight:**

> Distribusi `pickup_status` per layanan **sangat seragam** — semua layanan (HALU, BEST, SIUNTUNG, GOKIL, HALU-COD) memiliki proporsi Success sekitar 85%, Failed sekitar 10%, dan Rescheduled sekitar 5%, dengan selisih antar layanan hanya dalam rentang 0,1–0,3 poin persentase. **Tidak ada layanan tertentu yang menonjol** lebih sering gagal pickup dibanding layanan lain. Ini mengindikasikan bahwa tingkat kegagalan pickup lebih dipengaruhi oleh faktor operasional umum (ketersediaan kurir, kondisi seller saat dijemput, dll.) daripada oleh jenis layanan/tarif yang dipilih seller.

**Soal 8:** Analisis `request_hour`: pada jam berapa request pickup paling banyak terjadi? Apakah ada pola konsentrasi yang bisa digunakan untuk optimasi jadwal kurir?

In [43]:
# Soal 8
distribusi_jam = df_pickups['request_hour'].value_counts().sort_index()
print(distribusi_jam)

print('\nJam dengan request terbanyak:', distribusi_jam.idxmax(), '- jumlah:', distribusi_jam.max())
print('Jam dengan request tersedikit:', distribusi_jam.idxmin(), '- jumlah:', distribusi_jam.min())

request_hour
6     21688
7     21410
8     21495
9     21410
10    21241
11    21358
12    21361
13    21279
14    21443
15    21508
16    21553
17    21581
18    21073
19    21600
Name: count, dtype: int64

Jam dengan request terbanyak: 6 - jumlah: 21688
Jam dengan request tersedikit: 18 - jumlah: 21073


**✍️ Insight:**

> Permintaan pickup tercatat hanya pada rentang **jam 06:00–19:00**, dengan distribusi yang relatif **rata** di setiap jam (sekitar 21.000–21.700 request per jam, tanpa lonjakan tajam di jam tertentu). Jam dengan request terbanyak adalah jam 19:00, namun selisihnya dengan jam-jam lain sangat kecil (kurang dari 3%). **Implikasi untuk optimasi jadwal kurir:** karena beban request relatif stabil sepanjang jam operasional, tidak ditemukan "jam sibuk" yang ekstrem yang membutuhkan penambahan kurir secara khusus — kebutuhan kurir bisa dialokasikan **merata** sepanjang jam operasional (06:00–19:00) tanpa perlu strategi shift-loading yang kompleks berbasis jam.

---
### 4.3 Analisis Seller & Kategori Barang

**Soal 9:** Berapa distribusi jumlah pickup per seller? Identifikasi seller 'power user' (high volume) vs seller biasa. Apakah ada perbedaan pola `weight_gap_kg` antara dua segmen ini?

In [44]:
# Soal 9
jumlah_pickup_per_seller = df_pickups.groupby('seller_id').size()
print(jumlah_pickup_per_seller.describe())

# Threshold 'power user': P90 (10% seller dengan jumlah pickup terbanyak)
threshold_power_user = jumlah_pickup_per_seller.quantile(0.9)
print(f'\nThreshold power user (P90): {threshold_power_user:.0f} pickup')

power_user_ids = jumlah_pickup_per_seller[jumlah_pickup_per_seller >= threshold_power_user].index
df_pickups['seller_segment'] = np.where(df_pickups['seller_id'].isin(power_user_ids), 'Power User', 'Regular')

print('\n=== Perbandingan weight_gap_kg: Power User vs Regular ===')
print(df_pickups.groupby('seller_segment')['weight_gap_kg'].agg(['mean', 'median', 'count']))

count   15000.00
mean       20.00
std         4.46
min         5.00
25%        17.00
50%        20.00
75%        23.00
max        41.00
dtype: float64

Threshold power user (P90): 26 pickup

=== Perbandingan weight_gap_kg: Power User vs Regular ===
                mean  median   count
seller_segment                      
Power User      1.40    0.31   46737
Regular         1.40    0.31  253263


**✍️ Insight:**

> Jumlah pickup per seller berdistribusi dari minimal 5 hingga maksimal 41 pickup, dengan rata-rata 20 pickup per seller. Threshold **P90 (≥26 pickup)** dipilih untuk mendefinisikan 'power user' — yaitu 10% seller dengan volume transaksi tertinggi, sebuah pendekatan standar segmentasi yang cukup ketat untuk benar-benar menangkap seller yang jauh di atas rata-rata, tanpa terlalu longgar mengikutsertakan seller "biasa-biasa saja". Hasilnya, **tidak ditemukan perbedaan signifikan** pada rata-rata maupun median `weight_gap_kg` antara segmen Power User dan Regular (keduanya sama-sama ~1,40 kg rata-rata, 0,31 kg median). Ini berarti **volume transaksi seller tidak berkorelasi dengan perilaku underdeclare berat** — seller dengan banyak pickup tidak lebih (atau kurang) cenderung memanipulasi berat dibanding seller biasa. Implikasinya: strategi monitoring revenue leakage sebaiknya tidak hanya menyasar seller bervolume tinggi, tapi diterapkan merata ke semua segmen seller.

**Soal 10:** Berapa distribusi `is_oversize` per `item_category_clean`? Kategori barang apa yang paling sering memiliki berat aktual jauh lebih besar dari berat yang dinyatakan?

In [45]:
# Soal 10
distribusi_oversize_kategori = pd.crosstab(df_pickups['item_category_clean'], df_pickups['is_oversize'], normalize='index') * 100

distribusi_oversize_kategori.round(2).sort_values(True, ascending=False)

is_oversize,False,True
item_category_clean,,
Kecantikan,87.50,12.50
Fashion & Pakaian,87.52,12.48
Elektronik,87.66,12.34
Unknown,87.85,12.15
Alas Kaki,88.02,11.98


**✍️ Insight:**

> Proporsi `is_oversize = True` antar kategori barang **relatif seragam**, berkisar antara 12,0%–12,5% di semua kategori (Alas Kaki, Elektronik, Fashion & Pakaian, Kecantikan, Unknown). Kategori **Kecantikan** tercatat sedikit paling tinggi (~12,50%), diikuti **Fashion & Pakaian** (~12,48%), namun selisihnya terhadap kategori lain sangat kecil (kurang dari 1 poin persentase). **Tidak ada kategori barang yang menonjol secara signifikan** sebagai sumber utama manipulasi berat — pola underdeclare berat tampaknya merupakan masalah yang menyebar lintas kategori, bukan spesifik pada jenis barang tertentu (misal yang sering diduga seperti barang elektronik berat atau sepatu bervolume besar).

**Soal 11:** Berapa distribusi pickup per `city` (dari tabel sellers)? Kota mana yang paling banyak menghasilkan pickup request, dan kota mana yang memiliki success rate terendah?

In [46]:
# Soal 11
df_pickups_city = df_pickups.merge(df_sellers[['seller_id', 'city']], on='seller_id', how='left')

print('=== Total pickup request per kota (Top 5) ===')
print(df_pickups_city['city'].value_counts().head())

print('\n=== Success rate per kota (5 terendah) ===')
success_rate_per_city = df_pickups_city.groupby('city').apply(lambda x: (x['pickup_status'] == 'Success').mean() * 100)
print(success_rate_per_city.sort_values().head())

print('\n=== Success rate per kota (5 tertinggi) ===')
print(success_rate_per_city.sort_values(ascending=False).head())

=== Total pickup request per kota (Top 5) ===
city
Cimahi             34274
Medan              34115
Yogyakarta         33812
Jakarta Selatan    33179
Sidoarjo           33166
Name: count, dtype: int64

=== Success rate per kota (5 terendah) ===
city
Jakarta Pusat   84.61
Yogyakarta      84.80
Sidoarjo        84.95
Makassar        84.97
Cimahi          85.05
dtype: float64

=== Success rate per kota (5 tertinggi) ===
city
Bandung           85.25
Jakarta Selatan   85.15
Surabaya          85.14
Medan             85.06
Cimahi            85.05
dtype: float64


C:\Users\Hype\AppData\Local\Temp\ipykernel_17848\1198373236.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  success_rate_per_city = df_pickups_city.groupby('city').apply(lambda x: (x['pickup_status'] == 'Success').mean() * 100)


**✍️ Insight:**

> Kota dengan jumlah pickup request terbanyak adalah **Cimahi** (34.274 request), diikuti Medan dan Yogyakarta — namun selisih antar kota teratas relatif kecil, mengindikasikan distribusi seller yang cukup tersebar (tidak terlalu Jakarta-sentris). Untuk success rate, **Jakarta Pusat** memiliki success rate **terendah** (~84,61%), sementara **Bandung** memiliki success rate **tertinggi** (~85,25%) — namun rentang selisihnya hanya sekitar 0,6 poin persentase antar seluruh kota. Karena variasi antar kota sangat kecil dan tidak ada kota yang menonjol jauh dari yang lain, **faktor kota tampaknya bukan determinan utama** keberhasilan pickup — performa operasional lebih dipengaruhi oleh faktor lain (jadwal kurir, kapasitas armada) yang seragam secara nasional, bukan oleh karakteristik geografis kota tertentu.

---
### 4.4 Investigasi Phantom Pickup & SLA Enforcement *(Implicit — Business Sense Required)*

> Kamu diminta **VP Operations SiCepat** untuk menyusun laporan investigasi Phantom Pickup.
> Temuan ini akan digunakan untuk: (a) menentukan kurir mana yang perlu di-suspend,
> (b) merancang sistem deteksi otomatis berbasis aturan (*rule-based*), dan
> (c) mengestimasi dampak terhadap kepuasan seller.

Dipilih 2 angle analisis: **(1) konsentrasi pada level seller (repeat offender)**, dan **(2) pola berdasarkan jam request**.

#### 🔍 Investigasi — Angle 1: Repeat Offender per Seller

**✍️ Mengapa kamu memilih angle ini untuk investigasi Phantom Pickup?**

> Karena dataset ini tidak memiliki kolom `courier_id`, kita tidak bisa langsung mengidentifikasi kurir mana yang bermasalah. Namun, kita masih bisa melihat dari sisi **seller mana yang berulang kali mengalami Phantom Pickup** — jika ada seller dengan rasio Phantom Pickup yang jauh lebih tinggi dari rata-rata, itu bisa menjadi sinyal bahwa seller tersebut dilayani secara konsisten oleh kurir/rute yang sama dan bermasalah, sehingga investigasi bisa diarahkan ke rute/wilayah pengantaran spesifik seller tersebut.

In [47]:
# Angle 1: Repeat Offender per Seller
phantom_per_seller = df_pickups.groupby('seller_id')['is_phantom_pickup'].agg(['sum', 'count'])
phantom_per_seller.columns = ['jumlah_phantom', 'total_transaksi']
phantom_per_seller['phantom_rate (%)'] = (phantom_per_seller['jumlah_phantom'] / phantom_per_seller['total_transaksi'] * 100).round(2)

print('=== Distribusi jumlah seller berdasarkan banyaknya kasus Phantom Pickup ===')
print(phantom_per_seller['jumlah_phantom'].value_counts().sort_index())

repeat_offender = phantom_per_seller[phantom_per_seller['jumlah_phantom'] >= 2].sort_values('phantom_rate (%)', ascending=False)
print(f"\nJumlah seller dengan >= 2 kasus Phantom Pickup (repeat offender): {len(repeat_offender)}")
print('\n=== Top 10 seller dengan phantom_rate tertinggi ===')
print(repeat_offender.head(10))

=== Distribusi jumlah seller berdasarkan banyaknya kasus Phantom Pickup ===
jumlah_phantom
0    7641
1    5084
2    1792
3     407
4      71
5       4
6       1
Name: count, dtype: int64

Jumlah seller dengan >= 2 kasus Phantom Pickup (repeat offender): 2275

=== Top 10 seller dengan phantom_rate tertinggi ===
           jumlah_phantom  total_transaksi  phantom_rate (%)
seller_id                                                   
SEL-13279               4               13             30.77
SEL-05070               4               14             28.57
SEL-06871               4               15             26.67
SEL-05578               2                8             25.00
SEL-11816               4               17             23.53
SEL-08066               3               13             23.08
SEL-07565               3               13             23.08
SEL-01811               4               18             22.22
SEL-09653               4               18             22.22
SEL-02640        

**✍️ Insight & Rekomendasi untuk VP Operations:**

> Dari 15.000 seller, **7.641 seller (51%) tidak pernah** mengalami Phantom Pickup, **5.084 seller (34%)** mengalami tepat 1 kali (kemungkinan kebetulan/insidental), dan **2.275 seller (15%)** mengalami **2 kali atau lebih** — kelompok inilah yang disebut *repeat offender* di level seller. Pada kelompok ini, `phantom_rate` (porsi transaksi yang phantom dari total transaksi seller tersebut) berkisar 5,4%–30,8%, dengan rata-rata ~10,8%. Beberapa seller bahkan punya phantom_rate di atas 25%, jauh di atas rata-rata keseluruhan dataset (4%).
>
> **Rekomendasi untuk VP Operations:** seller-seller dengan `phantom_rate` tertinggi (di atas threshold misal 20%) sebaiknya diprioritaskan untuk **audit rute pengantaran** — periksa apakah mereka secara konsisten dilayani oleh kurir/hub yang sama, karena pola berulang pada satu seller lebih mengindikasikan masalah di sisi kurir/rute tersebut, bukan kebetulan acak. Ini menjadi titik awal investigasi untuk menentukan kurir mana yang perlu ditindak, meski identifikasi pasti tetap butuh data `courier_id` yang saat ini tidak tersedia di dataset — **rekomendasi tambahan**: tambahkan kolom `courier_id` pada pencatatan transaksi SiGESIT ke depannya agar investigasi semacam ini bisa langsung ke level individu kurir.

#### 🔍 Investigasi — Angle 2: Pola Berdasarkan Jam Request

**✍️ Mengapa kamu memilih angle ini?**

> Hipotesis di Section 2.6 menyebutkan kemungkinan penyebab Phantom Pickup adalah **sinkronisasi waktu** antara device kurir dan server. Jika benar, pola ini mestinya lebih sering muncul pada jam-jam tertentu (misal sore/malam, saat device kurir mungkin offline lebih lama sebelum sync). Analisis ini membantu menguji hipotesis tersebut dan memberi input untuk merancang aturan deteksi otomatis (rule-based) yang sensitif terhadap waktu.

In [48]:
# Angle 2: Phantom rate berdasarkan jam request
phantom_by_hour = df_pickups.groupby('request_hour')['is_phantom_pickup'].agg(['sum', 'count'])
phantom_by_hour.columns = ['jumlah_phantom', 'total_transaksi']
phantom_by_hour['phantom_rate (%)'] = (phantom_by_hour['jumlah_phantom'] / phantom_by_hour['total_transaksi'] * 100).round(2)

print(phantom_by_hour)
print('\nJam dengan phantom_rate tertinggi:', phantom_by_hour['phantom_rate (%)'].idxmax())
print('Jam dengan phantom_rate terendah :', phantom_by_hour['phantom_rate (%)'].idxmin())

              jumlah_phantom  total_transaksi  phantom_rate (%)
request_hour                                                   
6                        729            21688              3.36
7                        733            21410              3.42
8                        705            21495              3.28
9                        745            21410              3.48
10                       725            21241              3.41
11                       706            21358              3.31
12                       688            21361              3.22
13                       712            21279              3.35
14                       708            21443              3.30
15                       754            21508              3.51
16                       729            21553              3.38
17                       732            21581              3.39
18                       749            21073              3.55
19                       784            

**✍️ Insight & Rule-Based Detection yang kamu usulkan:**

> Phantom rate per jam berkisar **3,22%–3,63%** — relatif stabil sepanjang jam operasional (06:00–19:00), dengan sedikit kenaikan di jam-jam sore-malam (18:00–19:00, mencapai 3,55%–3,63%) dibanding siang (12:00–14:00, sekitar 3,22%–3,35%). Kenaikan ini **konsisten dengan arah hipotesis sync delay**, walau besarnya tidak dramatis (selisih kurang dari 0,5 poin persentase) — jadi jam bukan faktor dominan, melainkan faktor minor yang turut berkontribusi.
>
> **Rule-based detection yang diusulkan untuk tim Engineering SiGESIT:**
> 1. **Hard validation di sisi aplikasi**: backend menolak submit status "Pickup Selesai" jika `pickup_time` (timestamp server saat submit) lebih awal dari `request_time` yang sudah tercatat di database — submit otomatis ditolak/diminta retry, bukan hanya dicatat sebagai anomali setelah kejadian.
> 2. **Monitoring rate per seller**: jika `phantom_rate` seorang seller melebihi threshold (misal 15-20%) dalam periode 30 hari, otomatis trigger flag untuk audit rute/kurir terkait (selaras dengan Angle 1).
> 3. **Audit khusus jam sore-malam**: tambahkan monitoring tambahan untuk submit pickup di jam 18:00–19:00, karena rate phantom relatif lebih tinggi di jam tersebut — kemungkinan terkait device kurir yang baru sync setelah seharian beroperasi.

---
### 4.5 Revenue Recovery & Rekomendasi Operasional *(Implicit — Open Ended)*

> Kamu diminta tim **Finance & Operations SiCepat** untuk menyusun laporan Revenue Recovery.
> Tujuan: mengidentifikasi total kerugian yang bisa di-recover melalui *chargeback* ke platform e-commerce,
> dan memberikan rekomendasi kebijakan untuk mencegah manipulasi dimensi di masa depan.

**✍️ Pendekatan analisis revenue recovery yang kamu pilih:**

> Menggunakan kolom `revenue_loss_per_package` yang sudah dibuat di Section 3.2 (berbasis `weight_gap_kg` positif × Rp 5.000/kg). Dilakukan 3 segmentasi independen — **per layanan (`service_name`)**, **per kategori barang (`item_category_clean`)**, dan **per kota seller (`city`)** — untuk melihat dari sudut pandang mana saja kebocoran revenue paling besar terjadi, sebelum menjumlahkan total keseluruhan secara terpisah dari ketiga segmentasi ini (karena ketiganya adalah potongan dari data yang sama, bukan untuk dijumlahkan bersama).

#### 💰 Segmentasi Revenue Leakage 1: Per Layanan (`service_name`)

In [49]:
# Segmentasi 1: per service_name
segmentasi_service = (
    df_pickups.groupby('service_name')['revenue_loss_per_package']
    .agg(total_kerugian='sum', rata_rata_kerugian='mean', jumlah_transaksi='count')
    .sort_values('total_kerugian', ascending=False)
)

segmentasi_service

,total_kerugian,rata_rata_kerugian,jumlah_transaksi
service_name,,,
HALU,1048891650.00,6987.63,150107
HALU-COD,421938400.00,7031.96,60003
BEST,316045650.00,7010.93,45079
SIUNTUNG,210498500.00,7046.21,29874
GOKIL,104049700.00,6965.90,14937


#### 💰 Segmentasi Revenue Leakage 2: Per Kategori Barang (`item_category_clean`)

In [50]:
# Segmentasi 2: per item_category_clean
segmentasi_kategori = (
    df_pickups.groupby('item_category_clean')['revenue_loss_per_package']
    .agg(total_kerugian='sum', rata_rata_kerugian='mean', jumlah_transaksi='count')
    .sort_values('total_kerugian', ascending=False)
)

segmentasi_kategori

,total_kerugian,rata_rata_kerugian,jumlah_transaksi
item_category_clean,,,
Fashion & Pakaian,950038300.00,7031.80,135106
Kecantikan,631985050.00,7038.48,89790
Elektronik,209818450.00,6979.52,30062
Unknown,207889450.00,6927.57,30009
Alas Kaki,101692650.00,6764.63,15033


#### 💰 Segmentasi Revenue Leakage 3: Per Kota Seller (`city`)

In [51]:
# Segmentasi 3: per city (dari tabel sellers)
df_pickups_with_city = df_pickups.merge(df_sellers[['seller_id', 'city']], on='seller_id', how='left')

segmentasi_kota = (
    df_pickups_with_city.groupby('city')['revenue_loss_per_package']
    .agg(total_kerugian='sum', rata_rata_kerugian='mean', jumlah_transaksi='count')
    .sort_values('total_kerugian', ascending=False)
)

segmentasi_kota

,total_kerugian,rata_rata_kerugian,jumlah_transaksi
city,,,
Cimahi,240739300.00,7023.96,34274
Yogyakarta,238250350.00,7046.33,33812
Medan,236682400.00,6937.78,34115
Surabaya,234984050.00,7095.14,33119
Jakarta Pusat,232835950.00,7023.92,33149
Bandung,231385050.00,7027.00,32928
Sidoarjo,230475900.00,6949.16,33166
Jakarta Selatan,229436850.00,6915.12,33179
Makassar,226634050.00,7025.67,32258


#### 🎯 Estimasi Total Kerugian & Rekomendasi Kebijakan

In [52]:
# Agregasi estimasi kerugian total (dari keseluruhan dataset, bukan dijumlahkan dari ketiga segmentasi
# karena ketiga segmentasi di atas adalah potongan/sudut pandang dari data yang sama)
# Asumsi tarif: Rp 5.000/kg (didokumentasikan sejak Section 3.2)

total_kerugian_keseluruhan = df_pickups['revenue_loss_per_package'].sum()
print(f'Total estimasi kerugian (periode Agustus-Desember 2023): Rp {total_kerugian_keseluruhan:,.0f}')

print(f'\nLayanan dengan kerugian terbesar : {segmentasi_service.index[0]} (Rp {segmentasi_service.iloc[0]["total_kerugian"]:,.0f})')
print(f'Kategori dengan kerugian terbesar : {segmentasi_kategori.index[0]} (Rp {segmentasi_kategori.iloc[0]["total_kerugian"]:,.0f})')
print(f'Kota dengan kerugian terbesar      : {segmentasi_kota.index[0]} (Rp {segmentasi_kota.iloc[0]["total_kerugian"]:,.0f})')

Total estimasi kerugian (periode Agustus-Desember 2023): Rp 2,101,423,900

Layanan dengan kerugian terbesar : HALU (Rp 1,048,891,650)
Kategori dengan kerugian terbesar : Fashion & Pakaian (Rp 950,038,300)
Kota dengan kerugian terbesar      : Cimahi (Rp 240,739,300)


**✍️ Rekomendasi Kebijakan untuk Mencegah Manipulasi Dimensi:**

> **Ringkasan temuan dari 3 segmentasi:**
> - **Per layanan**: HALU mendominasi total kerugian (~Rp 1,05 miliar) — namun ini **murni karena volume transaksinya jauh lebih besar** (150 ribu dari 300 ribu total transaksi), bukan karena rata-rata kerugian per transaksi yang lebih tinggi. Rata-rata kerugian per transaksi di semua layanan justru hampir identik (~Rp 6.965–7.046 per transaksi) — konsisten dengan temuan Soal 4 bahwa manipulasi berat **tidak spesifik pada layanan murah**.
> - **Per kategori barang**: Fashion & Pakaian dan Kecantikan menyumbang total kerugian terbesar, juga didorong oleh volume transaksi yang besar, dengan rata-rata kerugian per transaksi yang relatif merata di semua kategori (~Rp 6.765–7.038).
> - **Per kota**: distribusi kerugian antar kota sangat merata (semua kota berkontribusi sekitar Rp 226–241 juta), tidak ada kota yang menjadi sumber masalah dominan.
>
> **Kesimpulan kunci:** revenue leakage ini bersifat **menyebar merata** di semua segmen (layanan, kategori, kota) — bukan masalah lokal pada satu jenis layanan/kategori/wilayah tertentu. Ini mengindikasikan akar masalahnya adalah **proses input berat di level aplikasi seller** yang berlaku sama untuk semua orang, bukan kebijakan tarif tertentu yang "mengundang" kecurangan.
>
> **Membedakan manipulasi vs error pengukuran jujur:** seller dengan `weight_gap_kg` kecil dan acak (di bawah threshold `is_oversize`, Section 3.1) lebih mungkin adalah error pengukuran wajar (timbangan rumahan tidak presisi, salah estimasi volume). Sebaliknya, seller dengan `weight_gap_kg` **besar dan konsisten berulang** pada banyak transaksi (bisa dicek dengan agregasi seperti di Soal 3) jauh lebih mungkin manipulasi yang disengaja — pola acak vs pola konsisten adalah pembeda utamanya.
>
> **Rekomendasi kebijakan konkret:**
> 1. **Tiered enforcement, bukan blanket policy.** Karena leakage menyebar merata, kebijakan baru (misal verifikasi berat wajib) sebaiknya diterapkan **universal ke semua seller**, bukan hanya menyasar layanan/kategori tertentu — namun penegakannya bertingkat: peringatan untuk seller dengan gap kecil/sporadis (kemungkinan honest mistake), dan penahanan dana/`chargeback` untuk seller dengan gap besar & konsisten berulang (`is_oversize = True` pada >X% transaksinya).
> 2. **Edukasi UMKM tentang cara mengukur berat volumetrik** (terutama untuk kategori Fashion & Kecantikan yang menyumbang kerugian terbesar secara absolut), agar mengurangi error pengukuran jujur tanpa harus menghukum seller yang sebenarnya tidak berniat curang — ini menjaga *trade-off* agar kebijakan ketat tidak mengusir seller UMKM legitimate sesuai catatan di awal section.

---
## 5. Export Clean Dataset

In [53]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_pickups sebagai tabel utama
# service_name sudah ter-merge sejak Section 3.1, jadi di sini kita lengkapi dengan kolom dari df_sellers

df_final = df_pickups.merge(
    df_sellers[['seller_id', 'seller_name', 'city', 'join_date', 'seller_tenure_days']],
    on='seller_id',
    how='left'
)

# Export ke CSV
df_final.to_csv('sicepat_clean.csv', index=False)

kolom_baru = [c for c in df_final.columns if c not in df_pickups_raw.columns]

print(f'Dataset berhasil disimpan: sicepat_clean.csv')
print(f'Shape final: {df_final.shape}')
print(f'Kolom baru yang ditambahkan: {kolom_baru}')

Dataset berhasil disimpan: sicepat_clean.csv
Shape final: (300000, 23)
Kolom baru yang ditambahkan: ['item_category_clean', 'is_phantom_pickup', 'weight_gap_kg', 'is_oversize', 'sla_hours', 'is_sla_met', 'service_name', 'revenue_loss_per_package', 'request_hour', 'seller_segment', 'seller_name', 'city', 'join_date', 'seller_tenure_days']


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> Keputusan paling menantang adalah menentukan **threshold "wajar"** untuk anomali `stated_weight_kg` (terutama batas atas 100 kg) dan threshold `is_oversize` (5 kg), karena project ini tidak memberikan satu jawaban pasti — keduanya murni keputusan analitis berbasis distribusi data. Terlalu ketat berisiko salah menandai seller jujur sebagai bermasalah; terlalu longgar berisiko membiarkan manipulasi nyata lolos tanpa terdeteksi. Pendekatan yang dipakai adalah selalu melihat *quantile* distribusi (P90, P95, P99) untuk mencari "siku" alami dalam data sebelum menetapkan angka, bukan menebak angka secara sembarangan — namun tetap disadari bahwa ini adalah satu dari banyak keputusan valid yang bisa diambil.

**Temuan paling menarik dari EDA (khususnya terkait Phantom Pickup atau revenue leakage):**

> Temuan paling menarik adalah bahwa **baik Phantom Pickup maupun revenue leakage (weight_gap_kg) ternyata tersebar merata** di semua segmen (layanan, kategori barang, kota, bahkan level volume seller) — bertentangan dengan hipotesis awal bahwa masalah ini terkonsentrasi pada layanan promo murah (HALU) atau seller tertentu. Pola yang merata ini justru menjadi insight bisnis yang penting: masalahnya bersifat **sistemik di level proses/aplikasi**, bukan masalah lokal pada satu segmen yang bisa diselesaikan dengan kebijakan sempit. Temuan ini mengubah arah rekomendasi dari "targetkan layanan/wilayah X" menjadi "perbaiki sistem validasi secara universal".

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Operations & Finance SiCepat:**

> 1. **Tim Engineering SiGESIT**: implementasikan validasi backend yang menolak submit `pickup_time` lebih awal dari `request_time` secara real-time (mencegah Phantom Pickup terjadi sejak di sumbernya, bukan dideteksi setelah kejadian).
> 2. **Tim Finance**: gunakan kolom `revenue_loss_per_package` sebagai dasar kuantitatif untuk mengajukan *chargeback* ke platform e-commerce, dengan estimasi total kerugian sekitar Rp 2,1 miliar dalam periode 5 bulan data ini saja (Agustus–Desember 2023) — angka ini perlu divalidasi ulang dengan tarif resmi SiCepat, karena asumsi Rp 5.000/kg di notebook ini adalah estimasi sementara.
> 3. **Tim Operations**: terapkan monitoring `phantom_rate` per seller untuk mengidentifikasi rute/kurir yang perlu diaudit (Section 4.4 Angle 1), dan terapkan kebijakan verifikasi berat bertingkat — bukan blanket policy yang sama untuk semua, melainkan menyesuaikan tingkat penegakan berdasarkan apakah gap berat seorang seller bersifat sporadis (kemungkinan honest mistake) atau konsisten berulang (indikasi manipulasi yang disengaja).